In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Carpeta del Día 12.
CARPETA_DATOS = Path("datos_dia_11")
CARPETA_DATOS.mkdir(exist_ok=True)

# Base de datos del Día 12.
RUTA_DB = CARPETA_DATOS / "consultas_join.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_11

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_11\consultas_join.db


In [2]:
def obtener_conexion():
    """
    Crea una conexión con SQLite y activa las llaves foráneas.
    """
    conexion = sqlite3.connect(RUTA_DB)
    conexion.execute("PRAGMA foreign_keys = ON")
    return conexion


conexion = obtener_conexion()

estado_fk = pd.read_sql_query("""
PRAGMA foreign_keys
""", conexion)

conexion.close()

estado_fk

,foreign_keys
0,1


In [3]:
conexion = obtener_conexion()
cursor = conexion.cursor()

# Eliminamos primero las tablas hijas.
cursor.execute("DROP TABLE IF EXISTS detalle_ventas")
cursor.execute("DROP TABLE IF EXISTS ventas")
cursor.execute("DROP TABLE IF EXISTS productos")
cursor.execute("DROP TABLE IF EXISTS clientes")

cursor.execute("""
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    telefono TEXT,
    ciudad TEXT DEFAULT 'No especificada'
)
""")

cursor.execute("""
CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT NOT NULL UNIQUE,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
)
""")

cursor.execute("""
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER NOT NULL,
    fecha TEXT NOT NULL,
    total REAL NOT NULL DEFAULT 0 CHECK(total >= 0),

    FOREIGN KEY (id_cliente)
        REFERENCES clientes(id_cliente)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
)
""")

cursor.execute("""
CREATE TABLE detalle_ventas (
    id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0),
    subtotal REAL NOT NULL CHECK(subtotal >= 0),

    FOREIGN KEY (id_venta)
        REFERENCES ventas(id_venta)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,

    FOREIGN KEY (id_producto)
        REFERENCES productos(id_producto)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
)
""")

conexion.commit()
conexion.close()

print("Tablas creadas correctamente.")

Tablas creadas correctamente.


In [4]:
conexion = obtener_conexion()
cursor = conexion.cursor()

clientes = [
    ("Ana López", "ana@example.com", "555-111-2222", "CDMX"),
    ("Carlos Pérez", "carlos@example.com", "555-333-4444", "Guadalajara"),
    ("María Torres", "maria@example.com", None, "Monterrey"),
    ("Luis Romero", "luis@example.com", "555-888-9999", "Puebla"),
    ("Sofía Herrera", "sofia@example.com", None, "Querétaro")
]

productos = [
    ("P001", "Mouse inalámbrico", "Accesorios", 249.90, 15, 1),
    ("P002", "Teclado mecánico", "Accesorios", 899.00, 8, 1),
    ("P003", "Monitor 24 pulgadas", "Pantallas", 3299.00, 4, 1),
    ("P004", "Cable HDMI", "Cables", 120.00, 20, 1),
    ("P005", "Memoria USB 64GB", "Almacenamiento", 150.00, 30, 1),
    ("P006", "Bocinas Bluetooth", "Audio", 699.00, 10, 1),
    ("P007", "Webcam HD", "Accesorios", 550.00, 6, 1),
    ("P008", "Producto sin ventas", "General", 100.00, 5, 1)
]

cursor.executemany("""
INSERT INTO clientes (nombre, correo, telefono, ciudad)
VALUES (?, ?, ?, ?)
""", clientes)

cursor.executemany("""
INSERT INTO productos (sku, nombre, categoria, precio, stock, activo)
VALUES (?, ?, ?, ?, ?, ?)
""", productos)

conexion.commit()
conexion.close()

print("Clientes y productos insertados correctamente.")

Clientes y productos insertados correctamente.


In [5]:
conexion = obtener_conexion()
cursor = conexion.cursor()

ventas = [
    # id_cliente, fecha, total
    (1, "2026-07-01", 1398.80),
    (2, "2026-07-02", 3539.00),
    (1, "2026-07-03", 570.00),
    (3, "2026-07-04", 699.00)
]

cursor.executemany("""
INSERT INTO ventas (id_cliente, fecha, total)
VALUES (?, ?, ?)
""", ventas)

detalles = [
    # id_venta, id_producto, cantidad, precio_unitario, subtotal
    (1, 1, 2, 249.90, 499.80),
    (1, 2, 1, 899.00, 899.00),
    (2, 3, 1, 3299.00, 3299.00),
    (2, 4, 2, 120.00, 240.00),
    (3, 4, 1, 120.00, 120.00),
    (3, 5, 3, 150.00, 450.00),
    (4, 6, 1, 699.00, 699.00)
]

cursor.executemany("""
INSERT INTO detalle_ventas (
    id_venta,
    id_producto,
    cantidad,
    precio_unitario,
    subtotal
)
VALUES (?, ?, ?, ?, ?)
""", detalles)

conexion.commit()
conexion.close()

print("Ventas y detalles insertados correctamente.")

Ventas y detalles insertados correctamente.


In [6]:
conexion = obtener_conexion()

print("Clientes")
display(pd.read_sql_query("""
SELECT *
FROM clientes
ORDER BY id_cliente
""", conexion))

print("Productos")
display(pd.read_sql_query("""
SELECT *
FROM productos
ORDER BY id_producto
""", conexion))

print("Ventas")
display(pd.read_sql_query("""
SELECT *
FROM ventas
ORDER BY id_venta
""", conexion))

print("Detalle de ventas")
display(pd.read_sql_query("""
SELECT *
FROM detalle_ventas
ORDER BY id_detalle
""", conexion))

conexion.close()

Clientes


,id_cliente,nombre,correo,telefono,ciudad
0,1,Ana López,ana@example.com,555-111-2222,CDMX
1,2,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara
2,3,María Torres,maria@example.com,NaN,Monterrey
3,4,Luis Romero,luis@example.com,555-888-9999,Puebla
4,5,Sofía Herrera,sofia@example.com,NaN,Querétaro


Productos


,id_producto,sku,nombre,categoria,precio,stock,activo
0,1,P001,Mouse inalámbrico,Accesorios,249.9,15,1
1,2,P002,Teclado mecánico,Accesorios,899.0,8,1
2,3,P003,Monitor 24 pulgadas,Pantallas,3299.0,4,1
3,4,P004,Cable HDMI,Cables,120.0,20,1
4,5,P005,Memoria USB 64GB,Almacenamiento,150.0,30,1
5,6,P006,Bocinas Bluetooth,Audio,699.0,10,1
6,7,P007,Webcam HD,Accesorios,550.0,6,1
7,8,P008,Producto sin ventas,General,100.0,5,1


Ventas


,id_venta,id_cliente,fecha,total
0,1,1,2026-07-01,1398.8
1,2,2,2026-07-02,3539.0
2,3,1,2026-07-03,570.0
3,4,3,2026-07-04,699.0


Detalle de ventas


,id_detalle,id_venta,id_producto,cantidad,precio_unitario,subtotal
0,1,1,1,2,249.9,499.8
1,2,1,2,1,899.0,899.0
2,3,2,3,1,3299.0,3299.0
3,4,2,4,2,120.0,240.0
4,5,3,4,1,120.0,120.0
5,6,3,5,3,150.0,450.0
6,7,4,6,1,699.0,699.0


In [ ]:
conexion = obtener_conexion()

ventas_sin_join = pd.read_sql_query("""
SELECT
    id_venta,
    id_cliente,
    fecha,
    total
FROM ventas
ORDER BY id_venta
""", conexion)

conexion.close()
{key: "ventas_sin_join", "value": ventas_sin_join}
ventas_sin_join

,id_venta,id_cliente,fecha,total
0,1,1,2026-07-01,1398.8
1,2,2,2026-07-02,3539.0
2,3,1,2026-07-03,570.0
3,4,3,2026-07-04,699.0


In [ ]:
conexion = obtener_conexion()

ventas_con_cliente = pd.read_sql_query("""
SELECT
    ventas.id_venta,
    ventas.fecha,
    ventas.id_cliente,
    clientes.nombre AS cliente,
    clientes.correo,
    ventas.total
FROM ventas
INNER JOIN clientes
    ON ventas.id_cliente = clientes.id_cliente
ORDER BY ventas.id_venta
""", conexion)

conexion.close()

ventas_con_cliente

,id_venta,fecha,id_cliente,cliente,correo,total
0,1,2026-07-01,1,Ana López,ana@example.com,1398.8
1,2,2026-07-02,2,Carlos Pérez,carlos@example.com,3539.0
2,3,2026-07-03,1,Ana López,ana@example.com,570.0
3,4,2026-07-04,3,María Torres,maria@example.com,699.0


In [9]:
conexion = obtener_conexion()

ventas_con_cliente_alias = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    v.id_cliente,
    c.nombre AS cliente,
    c.correo,
    v.total
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
ORDER BY v.id_venta
""", conexion)

conexion.close()

ventas_con_cliente_alias

,id_venta,fecha,id_cliente,cliente,correo,total
0,1,2026-07-01,1,Ana López,ana@example.com,1398.8
1,2,2026-07-02,2,Carlos Pérez,carlos@example.com,3539.0
2,3,2026-07-03,1,Ana López,ana@example.com,570.0
3,4,2026-07-04,3,María Torres,maria@example.com,699.0


In [10]:
conexion = obtener_conexion()

detalle_sin_join = pd.read_sql_query("""
SELECT
    id_detalle,
    id_venta,
    id_producto,
    cantidad,
    precio_unitario,
    subtotal
FROM detalle_ventas
ORDER BY id_venta, id_detalle
""", conexion)

conexion.close()

detalle_sin_join

,id_detalle,id_venta,id_producto,cantidad,precio_unitario,subtotal
0,1,1,1,2,249.9,499.8
1,2,1,2,1,899.0,899.0
2,3,2,3,1,3299.0,3299.0
3,4,2,4,2,120.0,240.0
4,5,3,4,1,120.0,120.0
5,6,3,5,3,150.0,450.0
6,7,4,6,1,699.0,699.0


In [11]:
conexion = obtener_conexion()

detalle_con_producto = pd.read_sql_query("""
SELECT
    dv.id_detalle,
    dv.id_venta,
    dv.id_producto,
    p.sku,
    p.nombre AS producto,
    p.categoria,
    dv.cantidad,
    dv.precio_unitario,
    dv.subtotal
FROM detalle_ventas dv
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY dv.id_venta, dv.id_detalle
""", conexion)

conexion.close()

detalle_con_producto

,id_detalle,id_venta,id_producto,sku,producto,categoria,cantidad,precio_unitario,subtotal
0,1,1,1,P001,Mouse inalámbrico,Accesorios,2,249.9,499.8
1,2,1,2,P002,Teclado mecánico,Accesorios,1,899.0,899.0
2,3,2,3,P003,Monitor 24 pulgadas,Pantallas,1,3299.0,3299.0
3,4,2,4,P004,Cable HDMI,Cables,2,120.0,240.0
4,5,3,4,P004,Cable HDMI,Cables,1,120.0,120.0
5,6,3,5,P005,Memoria USB 64GB,Almacenamiento,3,150.0,450.0
6,7,4,6,P006,Bocinas Bluetooth,Audio,1,699.0,699.0


In [12]:
conexion = obtener_conexion()

reporte_completo = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    c.ciudad,
    p.sku,
    p.nombre AS producto,
    p.categoria,
    dv.cantidad,
    dv.precio_unitario,
    dv.subtotal,
    v.total AS total_venta
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY v.id_venta, dv.id_detalle
""", conexion)

conexion.close()

reporte_completo

,id_venta,fecha,cliente,ciudad,sku,producto,categoria,cantidad,precio_unitario,subtotal,total_venta
0,1,2026-07-01,Ana López,CDMX,P001,Mouse inalámbrico,Accesorios,2,249.9,499.8,1398.8
1,1,2026-07-01,Ana López,CDMX,P002,Teclado mecánico,Accesorios,1,899.0,899.0,1398.8
2,2,2026-07-02,Carlos Pérez,Guadalajara,P003,Monitor 24 pulgadas,Pantallas,1,3299.0,3299.0,3539.0
3,2,2026-07-02,Carlos Pérez,Guadalajara,P004,Cable HDMI,Cables,2,120.0,240.0,3539.0
4,3,2026-07-03,Ana López,CDMX,P004,Cable HDMI,Cables,1,120.0,120.0,570.0
5,3,2026-07-03,Ana López,CDMX,P005,Memoria USB 64GB,Almacenamiento,3,150.0,450.0,570.0
6,4,2026-07-04,María Torres,Monterrey,P006,Bocinas Bluetooth,Audio,1,699.0,699.0,699.0


In [13]:
conexion = obtener_conexion()

ventas_ana = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    p.nombre AS producto,
    dv.cantidad,
    dv.subtotal
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
WHERE c.nombre = 'Ana López'
ORDER BY v.id_venta, dv.id_detalle
""", conexion)

conexion.close()

ventas_ana

,id_venta,fecha,cliente,producto,cantidad,subtotal
0,1,2026-07-01,Ana López,Mouse inalámbrico,2,499.8
1,1,2026-07-01,Ana López,Teclado mecánico,1,899.0
2,3,2026-07-03,Ana López,Cable HDMI,1,120.0
3,3,2026-07-03,Ana López,Memoria USB 64GB,3,450.0


In [14]:
conexion = obtener_conexion()

ventas_accesorios = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    p.nombre AS producto,
    p.categoria,
    dv.cantidad,
    dv.subtotal
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
WHERE p.categoria = 'Accesorios'
ORDER BY v.fecha, v.id_venta
""", conexion)

conexion.close()

ventas_accesorios

,id_venta,fecha,cliente,producto,categoria,cantidad,subtotal
0,1,2026-07-01,Ana López,Mouse inalámbrico,Accesorios,2,499.8
1,1,2026-07-01,Ana López,Teclado mecánico,Accesorios,1,899.0


In [15]:
conexion = obtener_conexion()

clientes_con_ventas_inner = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre AS cliente,
    v.id_venta,
    v.fecha,
    v.total
FROM clientes c
INNER JOIN ventas v
    ON c.id_cliente = v.id_cliente
ORDER BY c.id_cliente, v.id_venta
""", conexion)

conexion.close()

clientes_con_ventas_inner

,id_cliente,cliente,id_venta,fecha,total
0,1,Ana López,1,2026-07-01,1398.8
1,1,Ana López,3,2026-07-03,570.0
2,2,Carlos Pérez,2,2026-07-02,3539.0
3,3,María Torres,4,2026-07-04,699.0


In [16]:
conexion = obtener_conexion()

clientes_con_ventas_left = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre AS cliente,
    c.correo,
    v.id_venta,
    v.fecha,
    v.total
FROM clientes c
LEFT JOIN ventas v
    ON c.id_cliente = v.id_cliente
ORDER BY c.id_cliente, v.id_venta
""", conexion)

conexion.close()

clientes_con_ventas_left

,id_cliente,cliente,correo,id_venta,fecha,total
0,1,Ana López,ana@example.com,1.0,2026-07-01,1398.8
1,1,Ana López,ana@example.com,3.0,2026-07-03,570.0
2,2,Carlos Pérez,carlos@example.com,2.0,2026-07-02,3539.0
3,3,María Torres,maria@example.com,4.0,2026-07-04,699.0
4,4,Luis Romero,luis@example.com,NaN,NaN,NaN
5,5,Sofía Herrera,sofia@example.com,NaN,NaN,NaN


In [21]:
conexion = obtener_conexion()

clientes_sin_ventas = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre,
    c.correo,
    c.ciudad
FROM clientes c
INNER JOIN ventas v
    ON c.id_cliente = v.id_cliente
WHERE v.id_venta IS NULL
ORDER BY c.id_cliente
""", conexion)

conexion.close()

clientes_sin_ventas

,id_cliente,nombre,correo,ciudad


In [18]:
conexion = obtener_conexion()

productos_vendidos_inner = pd.read_sql_query("""
SELECT
    p.id_producto,
    p.sku,
    p.nombre AS producto,
    SUM(dv.cantidad) AS unidades_vendidas,
    SUM(dv.subtotal) AS importe_vendido
FROM productos p
INNER JOIN detalle_ventas dv
    ON p.id_producto = dv.id_producto
GROUP BY p.id_producto, p.sku, p.nombre
ORDER BY importe_vendido DESC
""", conexion)

conexion.close()

productos_vendidos_inner

,id_producto,sku,producto,unidades_vendidas,importe_vendido
0,3,P003,Monitor 24 pulgadas,1,3299.0
1,2,P002,Teclado mecánico,1,899.0
2,6,P006,Bocinas Bluetooth,1,699.0
3,1,P001,Mouse inalámbrico,2,499.8
4,5,P005,Memoria USB 64GB,3,450.0
5,4,P004,Cable HDMI,3,360.0


In [19]:
conexion = obtener_conexion()

productos_todos_left = pd.read_sql_query("""
SELECT
    p.id_producto,
    p.sku,
    p.nombre AS producto,
    COALESCE(SUM(dv.cantidad), 0) AS unidades_vendidas,
    COALESCE(SUM(dv.subtotal), 0) AS importe_vendido
FROM productos p
LEFT JOIN detalle_ventas dv
    ON p.id_producto = dv.id_producto
GROUP BY p.id_producto, p.sku, p.nombre
ORDER BY importe_vendido DESC
""", conexion)

conexion.close()

productos_todos_left

,id_producto,sku,producto,unidades_vendidas,importe_vendido
0,3,P003,Monitor 24 pulgadas,1,3299.0
1,2,P002,Teclado mecánico,1,899.0
2,6,P006,Bocinas Bluetooth,1,699.0
3,1,P001,Mouse inalámbrico,2,499.8
4,5,P005,Memoria USB 64GB,3,450.0
5,4,P004,Cable HDMI,3,360.0
6,7,P007,Webcam HD,0,0.0
7,8,P008,Producto sin ventas,0,0.0


In [20]:
conexion = obtener_conexion()

total_por_cliente = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre AS cliente,
    COUNT(v.id_venta) AS cantidad_ventas,
    COALESCE(SUM(v.total), 0) AS total_comprado
FROM clientes c
LEFT JOIN ventas v
    ON c.id_cliente = v.id_cliente
GROUP BY c.id_cliente, c.nombre
ORDER BY total_comprado DESC
""", conexion)

conexion.close()

total_por_cliente

,id_cliente,cliente,cantidad_ventas,total_comprado
0,2,Carlos Pérez,1,3539.0
1,1,Ana López,2,1968.8
2,3,María Torres,1,699.0
3,4,Luis Romero,0,0.0
4,5,Sofía Herrera,0,0.0
